# Day 2 — Notebook 3: Complete Solutions & Extensions
**Tax Tech Transformation Program | Big4 Firm**

This notebook contains:
- ✅ Exercise 2 solution: extended knowledge base (11 entries)
- ✅ Exercise 3 solution: `get_filing_deadline()` tool
- ✅ Enhanced `TaxFAQAgent` with audit trail export
- ✅ Bonus: Batch FAQ mode → CSV export

> **Facilitator note:** Share only *after* participants attempt Notebooks 1–2.

## Step 1: Install & Import

In [1]:
%pip install openai --quiet

import os
import json
import csv
import datetime
from getpass import getpass
from openai import AzureOpenAI

print("✅ Imports OK")


[notice] A new release of pip is available: 25.2 -> 26.0.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.
✅ Imports OK


## Step 2: Configure Azure OpenAI Client

In [2]:
# AZURE_OPENAI_ENDPOINT = input("Endpoint (e.g. https://xxx.openai.azure.com/): ").strip()
# AZURE_OPENAI_API_KEY  = getpass("API Key: ")
# AZURE_DEPLOYMENT_NAME = input("Deployment name (e.g. gpt-4o): ").strip()

# client = AzureOpenAI(
#     azure_endpoint = AZURE_OPENAI_ENDPOINT,
#     api_key        = AZURE_OPENAI_API_KEY,
#     api_version    = "2024-02-01",
# )
# print("✅ Azure OpenAI client initialised")

Alternatively, you can uncomment and run the following section to use .env variables.

In [3]:
from dotenv import load_dotenv

load_dotenv()

AZURE_OPENAI_ENDPOINT = os.getenv("AZURE_OPENAI_ENDPOINT")
AZURE_OPENAI_API_KEY = os.getenv("AZURE_OPENAI_API_KEY")
AZURE_DEPLOYMENT_NAME = os.getenv("AZURE_DEPLOYMENT_NAME")
AZURE_API_VERSION = os.getenv("AZURE_API_VERSION")

# Initialise client
client = AzureOpenAI(
    azure_endpoint = AZURE_OPENAI_ENDPOINT,
    api_key        = AZURE_OPENAI_API_KEY,
    api_version    = AZURE_API_VERSION,
)
print("✅ Azure OpenAI client initialised successfully!")

✅ Azure OpenAI client initialised successfully!


## Step 3: Complete Knowledge Base (Exercise 2 — Solution)

Extended from 8 to 11 entries, adding: new tax regime slabs, Section 44AD presumptive taxation, and GST registration threshold.

In [4]:
TAX_KB = {
    "gst_standard_rate": {
        "keywords": ["gst", "goods and services tax", "standard rate", "gst rate"],
        "answer": (
            "The standard GST rate in India is 18% for most services. "
            "Goods are taxed at 5%, 12%, 18%, or 28% depending on the category. "
            "Essential goods attract 0% or 5%; luxury and sin goods attract 28%."
        )
    },
    "tds_contractors": {
        "keywords": ["tds", "contractor", "section 194c", "194-c", "contractor payment"],
        "answer": (
            "Under Section 194C, TDS on contractor payments is 1% for individuals/HUF "
            "and 2% for companies. Threshold: Rs 30,000 per payment or "
            "Rs 1,00,000 aggregate per financial year."
        )
    },
    "corporate_tax_india": {
        "keywords": ["corporate tax", "company tax rate", "domestic company"],
        "answer": (
            "Domestic companies (turnover up to Rs 400 crore): 25% base rate. "
            "Others: 30%. New manufacturing companies (post Oct 2019): 15% under "
            "Section 115BAB. Effective rate approx 25.17% after surcharge and cess."
        )
    },
    "section_80c": {
        "keywords": ["80c", "section 80c", "deduction", "tax saving", "lic", "ppf", "elss"],
        "answer": (
            "Section 80C allows deductions up to Rs 1,50,000 per year. "
            "Eligible: LIC premiums, PPF, ELSS, NSC, 5-yr FDs, tuition fees, "
            "home loan principal, EPF. Available under Old Tax Regime only."
        )
    },
    "transfer_pricing": {
        "keywords": ["transfer pricing", "arm's length", "international transaction"],
        "answer": (
            "Transfer pricing (Sections 92-92F): transactions between associated "
            "enterprises must be at arm's length. Master File required if consolidated "
            "group revenue exceeds Rs 500 crore. Typical due date: October 31."
        )
    },
    "gst_input_tax_credit": {
        "keywords": ["itc", "input tax credit", "gst credit", "claim credit"],
        "answer": (
            "ITC allows deducting input GST from output GST payable. Conditions: "
            "supplier filed returns, buyer holds tax invoice, goods/services received, "
            "GST paid to government. Blocked credits (Rule 17) not eligible."
        )
    },
    "capital_gains_ltcg": {
        "keywords": ["capital gains", "ltcg", "long term", "equity", "section 112a"],
        "answer": (
            "LTCG on listed equity/equity MFs exceeding Rs 1 lakh: 10% without "
            "indexation (Section 112A). Holding periods: equity >12 months, "
            "immovable property >24 months, debt MFs >36 months."
        )
    },
    "advance_tax": {
        "keywords": ["advance tax", "installment", "quarterly tax"],
        "answer": (
            "Advance tax payable when liability exceeds Rs 10,000. Installments: "
            "15% (Jun 15), 45% (Sep 15), 75% (Dec 15), 100% (Mar 15). "
            "Interest under 234B/234C for shortfall. Senior citizens without "
            "business income are exempt."
        )
    },
    # Exercise 2 — solutions
    "new_tax_regime_slabs": {
        "keywords": ["new tax regime", "new regime", "income tax slab",
                     "fy 2025", "budget 2025", "2025 slab"],
        "answer": (
            "New Tax Regime FY 2025-26: Nil (up to Rs 3L), 5% (Rs 3-7L), "
            "10% (Rs 7-10L), 15% (Rs 10-12L), 20% (Rs 12-15L), 30% (above Rs 15L). "
            "Standard deduction Rs 75,000. No Chapter VI-A deductions. "
            "Rebate u/s 87A: zero tax for income up to Rs 7 lakh."
        )
    },
    "section_44ad": {
        "keywords": ["44ad", "section 44ad", "presumptive", "presumptive taxation"],
        "answer": (
            "Section 44AD: Presumptive taxation for businesses with turnover up to "
            "Rs 2 crore. Deemed profit: 8% of turnover (6% for digital receipts). "
            "No books or audit required. Must continue for 5 years once opted."
        )
    },
    "gst_registration_threshold": {
        "keywords": ["gst registration", "registration limit", "threshold", "gst limit"],
        "answer": (
            "GST registration mandatory when annual turnover exceeds Rs 20 lakh "
            "(services) or Rs 40 lakh (goods, most states). Special category "
            "states: Rs 10 lakh. Voluntary registration permitted below threshold."
        )
    },
}

print(f"✅ Knowledge base loaded: {len(TAX_KB)} entries")
for k in TAX_KB:
    print(f"  • {k}")

✅ Knowledge base loaded: 11 entries
  • gst_standard_rate
  • tds_contractors
  • corporate_tax_india
  • section_80c
  • transfer_pricing
  • gst_input_tax_credit
  • capital_gains_ltcg
  • advance_tax
  • new_tax_regime_slabs
  • section_44ad
  • gst_registration_threshold


## Step 4: All Three Tool Functions (Exercise 3 — Solution)

- `search_tax_knowledge(query)` — keyword search over TAX_KB
- `calculate_tax(amount, rate)` — rupee tax computation
- `get_filing_deadline(tax_type)` — returns Indian tax filing due dates

In [5]:
def search_tax_knowledge(query):
    # Search the tax knowledge base for the best matching entry
    query_lower = query.lower()
    matches = []
    for topic, data in TAX_KB.items():
        score = sum(1 for kw in data['keywords'] if kw in query_lower)
        if score > 0:
            matches.append((score, topic, data['answer']))
    if not matches:
        return (
            "No specific information found in the knowledge base. "
            "Recommend consulting official tax documentation."
        )
    matches.sort(reverse=True)
    _, best_topic, best_answer = matches[0]
    result = f"[Source: {best_topic}]\n{best_answer}"
    if len(matches) > 1:
        related = ', '.join(t for _, t, _ in matches[1:3])
        result += f"\nRelated topics: {related}"
    return result


def calculate_tax(amount, rate):
    # Calculate tax amount for a given base amount and rate percentage
    if amount < 0:
        return "Error: Amount cannot be negative."
    if not (0 <= rate <= 100):
        return "Error: Rate must be between 0 and 100."
    tax_amount = round(amount * rate / 100, 2)
    total      = round(amount + tax_amount, 2)
    return (
        f"Tax Calculation:\n"
        f"  Base   : Rs {amount:,.2f}\n"
        f"  Rate   : {rate}%\n"
        f"  Tax    : Rs {tax_amount:,.2f}\n"
        f"  Total  : Rs {total:,.2f}"
    )


def get_filing_deadline(tax_type):
    # Return the filing deadline for a specified Indian tax form or type
    deadlines = {
        "gstr-1":        "11th of following month (monthly); 13th of Q+1 (QRMP quarterly)",
        "gstr-3b":       "20th of following month (monthly); 22nd/24th for QRMP filers",
        "gstr-9":        "December 31 of the following financial year",
        "itr":           "July 31 (individuals, non-audit); October 31 (companies and audit)",
        "tds-return":    "Q1: Jul 31 | Q2: Oct 31 | Q3: Jan 31 | Q4: May 31",
        "advance-tax-q1":"June 15 — 15% of estimated annual liability",
        "advance-tax-q2":"September 15 — 45% cumulative",
        "advance-tax-q3":"December 15 — 75% cumulative",
        "advance-tax-q4":"March 15 — 100% cumulative",
        "form-15ca-cb":  "Before remittance of any foreign payment",
        "gstr-4":        "April 30 (annual return for composition dealers)",
    }
    key_input = tax_type.lower().strip()
    for key, deadline in deadlines.items():
        if key in key_input or key_input in key:
            return f"Filing deadline — {key.upper()}: {deadline}"
    return (
        f"Deadline for '{tax_type}' not found. "
        "Check incometaxindia.gov.in or gst.gov.in for latest dates."
    )


# Smoke tests
print("SEARCH:", search_tax_knowledge("new tax regime slabs 2025")[:80])
print()
print("CALC  :", calculate_tax(500000, 18))
print()
print("DUE   :", get_filing_deadline("GSTR-1"))
print("DUE   :", get_filing_deadline("TDS return"))

SEARCH: [Source: new_tax_regime_slabs]
New Tax Regime FY 2025-26: Nil (up to Rs 3L), 5% 

CALC  : Tax Calculation:
  Base   : Rs 500,000.00
  Rate   : 18%
  Tax    : Rs 90,000.00
  Total  : Rs 590,000.00

DUE   : Filing deadline — GSTR-1: 11th of following month (monthly); 13th of Q+1 (QRMP quarterly)
DUE   : Deadline for 'TDS return' not found. Check incometaxindia.gov.in or gst.gov.in for latest dates.


## Step 5: Complete Tools Schema — All 3 Tools

In [6]:
TOOLS = [
    {
        "type": "function",
        "function": {
            "name": "search_tax_knowledge",
            "description": (
                "Search the internal tax knowledge base for Indian tax regulations, "
                "rates, sections, and compliance requirements. Always call this before "
                "answering any factual tax question."
            ),
            "parameters": {
                "type": "object",
                "properties": {
                    "query": {
                        "type": "string",
                        "description": "Tax-related search query"
                    }
                },
                "required": ["query"]
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "calculate_tax",
            "description": "Calculate tax amount given a base rupee amount and percentage rate.",
            "parameters": {
                "type": "object",
                "properties": {
                    "amount": {"type": "number", "description": "Base amount in Rupees"},
                    "rate":   {"type": "number", "description": "Tax rate as a percentage"}
                },
                "required": ["amount", "rate"]
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "get_filing_deadline",
            "description": (
                "Retrieve the filing deadline for a specific Indian tax form or type "
                "(e.g. GSTR-1, GSTR-3B, ITR, TDS return, advance tax instalments)."
            ),
            "parameters": {
                "type": "object",
                "properties": {
                    "tax_type": {
                        "type": "string",
                        "description": "Tax form or type (e.g. GSTR-1, ITR company, TDS return)"
                    }
                },
                "required": ["tax_type"]
            }
        }
    }
]

TOOL_FUNCTIONS = {
    "search_tax_knowledge": search_tax_knowledge,
    "calculate_tax":        calculate_tax,
    "get_filing_deadline":  get_filing_deadline,
}

print(f"✅ {len(TOOLS)} tools registered: {list(TOOL_FUNCTIONS.keys())}")

✅ 3 tools registered: ['search_tax_knowledge', 'calculate_tax', 'get_filing_deadline']


## Step 6: Enhanced TaxFAQAgent

Key improvements over the exercise scaffold:
- Colour-coded ReAct trace (no triple-quote issues)
- `save_session()` exports full conversation + trace to JSON
- Clean `reset_memory()` clears both history and log

In [7]:
SYSTEM_PROMPT = (
    "You are TaxBot, an expert AI assistant for tax professionals at a Big4 firm.\n"
    "Rules:\n"
    "1. Always call search_tax_knowledge before answering any factual tax question.\n"
    "2. Use calculate_tax for any rupee computation.\n"
    "3. Use get_filing_deadline for any due date or deadline questions.\n"
    "4. Never fabricate statutory section numbers or rates.\n"
    "5. Qualify answers: as per current regulations, please verify for latest updates."
)


class TaxFAQAgent:
    # ReAct Tax FAQ Agent with multi-turn memory and colour-coded trace

    COLORS = {
        "thought":     "\033[94m",
        "action":      "\033[93m",
        "observation": "\033[92m",
        "answer":      "\033[96m",
        "user":        "\033[97m",
    }
    RESET = "\033[0m"

    def __init__(self, client, deployment_name, max_iterations=8, verbose=True):
        self.client               = client
        self.deployment_name      = deployment_name
        self.max_iterations       = max_iterations
        self.verbose              = verbose
        self.conversation_history = []
        self.session_log          = []

    def _log(self, label, content):
        color = self.COLORS.get(label.lower(), '')
        line  = f"{color}[{label.upper()}]{self.RESET} {str(content)}"
        if self.verbose:
            print(line)
        self.session_log.append({
            "timestamp": datetime.datetime.now().isoformat(),
            "label":     label.upper(),
            "content":   str(content)
        })

    def _execute_tool(self, tool_call):
        name = tool_call.function.name
        try:
            args = json.loads(tool_call.function.arguments)
        except json.JSONDecodeError:
            return f"Error: could not parse arguments for {name}"
        if name not in TOOL_FUNCTIONS:
            return f"Error: unknown tool '{name}'"
        self._log('action', f'Calling {name}({args})')
        result = TOOL_FUNCTIONS[name](**args)
        self._log('observation', result[:300])
        return result

    def chat(self, user_message):
        self._log('user', user_message)
        self.conversation_history.append({'role': 'user', 'content': user_message})
        for iteration in range(1, self.max_iterations + 1):
            self._log('thought', f'Iteration {iteration}/{self.max_iterations}')
            messages = [{'role': 'system', 'content': SYSTEM_PROMPT}] + self.conversation_history
            response = self.client.chat.completions.create(
                model       = self.deployment_name,
                messages    = messages,
                tools       = TOOLS,
                tool_choice = 'auto',
                temperature = 0.1,
                max_tokens  = 600,
            )
            msg = response.choices[0].message
            if msg.tool_calls:
                self.conversation_history.append({
                    'role':       'assistant',
                    'content':    msg.content or '',
                    'tool_calls': [
                        {
                            'id':       tc.id,
                            'type':     'function',
                            'function': {
                                'name':      tc.function.name,
                                'arguments': tc.function.arguments
                            }
                        }
                        for tc in msg.tool_calls
                    ]
                })
                for tc in msg.tool_calls:
                    result = self._execute_tool(tc)
                    self.conversation_history.append({
                        'role':         'tool',
                        'tool_call_id': tc.id,
                        'content':      result
                    })
            else:
                final = msg.content or '(No response generated)'
                self.conversation_history.append({'role': 'assistant', 'content': final})
                self._log('answer', final)
                print()
                return final
        fallback = 'Agent reached maximum iterations. Please simplify your question.'
        self.conversation_history.append({'role': 'assistant', 'content': fallback})
        return fallback

    def reset_memory(self):
        self.conversation_history = []
        self.session_log          = []
        if self.verbose:
            print('🔄 Memory cleared')

    def save_session(self, filename=None):
        ts       = datetime.datetime.now().strftime('%Y%m%d_%H%M%S')
        filename = filename or f'taxbot_session_{ts}.json'
        payload  = {
            'exported_at':  datetime.datetime.now().isoformat(),
            'model':        self.deployment_name,
            'turn_count':   sum(1 for m in self.conversation_history if m['role'] == 'user'),
            'history':      self.conversation_history,
            'trace_log':    self.session_log,
        }
        with open(filename, 'w', encoding='utf-8') as fh:
            json.dump(payload, fh, indent=2, default=str)
        print(f'✅ Session saved -> {filename}')
        return filename


agent = TaxFAQAgent(client=client, deployment_name=AZURE_DEPLOYMENT_NAME, verbose=True)
print("✅ Enhanced TaxFAQAgent ready — 3 tools")

✅ Enhanced TaxFAQAgent ready — 3 tools


## Step 7: Full Integration Test — All 3 Tools

Watch the `[THOUGHT]` → `[ACTION]` → `[OBSERVATION]` → `[ANSWER]` trace to see the ReAct loop in action.

In [8]:
test_queries = [
    "What are the income tax slabs under the new tax regime for FY 2025-26?",
    "Calculate GST at 18% on a service invoice of Rs 3,75,000",
    "When is the deadline for filing GSTR-3B?",
    "What is the TDS rate for contractors? Also calculate TDS on a payment of Rs 80,000",
    "When is the advance tax Q2 instalment due and what percentage?",
]

for query in test_queries:
    print("=" * 70)
    agent.reset_memory()
    agent.chat(query)

🔄 Memory cleared
[USER] What are the income tax slabs under the new tax regime for FY 2025-26?
[THOUGHT] Iteration 1/8
[ACTION] Calling search_tax_knowledge({'query': 'Income tax slabs under new tax regime for FY 2025-26'})
[OBSERVATION] [Source: new_tax_regime_slabs]
New Tax Regime FY 2025-26: Nil (up to Rs 3L), 5% (Rs 3-7L), 10% (Rs 7-10L), 15% (Rs 10-12L), 20% (Rs 12-15L), 30% (above Rs 15L). Standard deduction Rs 75,000. No Chapter VI-A deductions. Rebate u/s 87A: zero tax for income up to Rs 7 lakh.
[THOUGHT] Iteration 2/8
[ANSWER] As per current regulations for FY 2025-26 under the new tax regime:

- **Income up to Rs 3,00,000**: Nil
- **Income from Rs 3,00,001 to Rs 7,00,000**: 5%
- **Income from Rs 7,00,001 to Rs 10,00,000**: 10%
- **Income from Rs 10,00,001 to Rs 12,00,000**: 15%
- **Income from Rs 12,00,001 to Rs 15,00,000**: 20%
- **Income above Rs 15,00,000**: 30%

Additional details:
- **Standard deduction**: Rs 75,000.
- **Rebate under Section 87A**: Zero tax for income u

## Step 8: Save Session & Inspect Audit Trail

The saved JSON contains every message and every tool call — useful for demonstrating auditability to tax managers.

In [9]:
agent.reset_memory()
agent.chat("What is Section 80C?")
agent.chat("What is the maximum deduction amount allowed?")
agent.chat("If I am in the 30% slab and invest Rs 1,50,000, how much tax do I save?")

saved_file = agent.save_session()

with open(saved_file, encoding='utf-8') as fh:
    data = json.load(fh)

print(f"\nAudit trail summary:")
print(f"  Turns        : {data['turn_count']}")
print(f"  History msgs : {len(data['history'])}")
print(f"  Trace entries: {len(data['trace_log'])}")
print(f"\nTrace (first 5 lines):")
for entry in data['trace_log'][:5]:
    print(f"  [{entry['label']}] {entry['content'][:90]}")

🔄 Memory cleared
[USER] What is Section 80C?
[THOUGHT] Iteration 1/8
[ACTION] Calling search_tax_knowledge({'query': 'Section 80C of Indian Income Tax Act'})
[OBSERVATION] [Source: section_80c]
Section 80C allows deductions up to Rs 1,50,000 per year. Eligible: LIC premiums, PPF, ELSS, NSC, 5-yr FDs, tuition fees, home loan principal, EPF. Available under Old Tax Regime only.
[THOUGHT] Iteration 2/8
[ANSWER] Section 80C of the Indian Income Tax Act allows individuals and Hindu Undivided Families (HUFs) to claim deductions up to ₹1,50,000 per financial year. Eligible investments and expenses include:

- Life Insurance Premiums (LIC)
- Public Provident Fund (PPF)
- Equity Linked Savings Scheme (ELSS)
- National Savings Certificate (NSC)
- 5-year Fixed Deposits with banks
- Tuition fees for children
- Principal repayment of home loans
- Employee Provident Fund (EPF)

This deduction is available only under the Old Tax Regime. As per current regulations, please verify for the latest updates

## Bonus: Batch FAQ Mode — Export Q&A to CSV

Run a list of questions in silent mode and export to `tax_faq_output.csv`. Useful for building FAQ documents or testing knowledge base coverage.

In [10]:
def batch_faq(agent_instance, questions, output_file='tax_faq_output.csv'):
    # Process a list of questions silently and export Q&A pairs to CSV
    results = []
    for i, question in enumerate(questions, 1):
        print(f'[{i}/{len(questions)}] {question[:60]}...')
        agent_instance.reset_memory()
        answer = agent_instance.chat(question)
        results.append({'#': i, 'question': question, 'answer': answer})
    with open(output_file, 'w', newline='', encoding='utf-8') as fh:
        writer = csv.DictWriter(fh, fieldnames=['#', 'question', 'answer'])
        writer.writeheader()
        writer.writerows(results)
    print(f'\n✅ {len(results)} Q&A pairs saved -> {output_file}')
    return results


faq_questions = [
    "What is the GST rate for professional services?",
    "What is Section 80C and its deduction limit?",
    "What is the TDS rate for contractor payments?",
    "When is the ITR filing deadline for companies?",
    "How does input tax credit work under GST?",
    "What are the long-term capital gains tax rules for equity?",
    "What are the new income tax slabs for FY 2025-26?",
    "When is the GSTR-9 annual return due?",
]

silent_agent = TaxFAQAgent(client=client, deployment_name=AZURE_DEPLOYMENT_NAME, verbose=False)
results      = batch_faq(silent_agent, faq_questions)

print("\nFAQ Preview (first 2 entries):")
for r in results[:2]:
    print(f"Q{r['#']}: {r['question']}")
    print(f"A : {r['answer'][:200]}...")
    print()

[1/8] What is the GST rate for professional services?...

[2/8] What is Section 80C and its deduction limit?...

[3/8] What is the TDS rate for contractor payments?...

[4/8] When is the ITR filing deadline for companies?...

[5/8] How does input tax credit work under GST?...

[6/8] What are the long-term capital gains tax rules for equity?...

[7/8] What are the new income tax slabs for FY 2025-26?...

[8/8] When is the GSTR-9 annual return due?...


✅ 8 Q&A pairs saved -> tax_faq_output.csv

FAQ Preview (first 2 entries):
Q1: What is the GST rate for professional services?
A : As per current regulations, the GST rate for professional services in India is 18%. Please verify for the latest updates or specific exemptions....

Q2: What is Section 80C and its deduction limit?
A : As per current regulations, Section 80C of the Income Tax Act allows deductions up to ₹1,50,000 per financial year. This deduction is available for investments and expenses such as LIC premiums, PPF, ...

